# Spark Structured Streaming — Wikistreams (version enrichie)

Ce notebook reprend la démo du lab (modifications par heure, robots vs humains) puis ajoute les explorations des étapes 10 et 12.

| Section | Fenêtre | Ce qu'on mesure |
|---|---|---|
| Démo | fixe 1 h | modifications robots / humains |
| A | **fixe (tumbling)** 10 min | activité, contributeurs distincts, octets ajoutés / retirés |
| B | **glissante (overlapping)** 10 min, pas de 5 min | volume, part des robots, taille moyenne |
| B' | fixe vs glissante | vérification : chaque événement compte 2 fois en glissante |
| C | fixe 15 min + requête sans état | classes de taille, alertes sur les grosses modifications |
| D | fixe 30 min | top 3 des pages les plus modifiées |
| E | glissante 1 h, pas de 15 min + watermark | même logique en mode `update` |
| F | fixe 5 min | activité par wiki (si le producteur suit plusieurs langues) |

**Où voir les résultats ?** Les sorties `console` partent dans les logs du conteneur :
`docker logs -f pyspark_notebook`. Les sorties `memory` (B' et D) s'affichent directement ici.

**Conseil :** lance une exploration à la fois. Chaque cellule de lancement arrête d'abord les requêtes en cours, sinon les tableaux de plusieurs requêtes se mélangent dans les logs.

## 1. Imports et session Spark

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, avg, when, coalesce, lit, substring,
    approx_count_distinct, timestamp_seconds,
    sum as _sum, max as _max, abs as _abs, round as _round,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, BooleanType,
)

In [2]:
# Le connecteur Kafka doit avoir EXACTEMENT la même version que Spark.
# La démo figeait 4.1.0 : on lit plutôt la version présente dans l'image.
SPARK_VERSION = pyspark.__version__
KAFKA_PACKAGE = f"org.apache.spark:spark-sql-kafka-0-10_2.13:{SPARK_VERSION}"
print("Spark", SPARK_VERSION, "->", KAFKA_PACKAGE)

spark = (
    SparkSession.builder
    .appName("KafkaStreamingPrototype")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    # 200 partitions par défaut : beaucoup trop pour un petit flux local,
    # chaque micro-batch avec état serait inutilement lent
    .config("spark.sql.shuffle.partitions", "4")
    # Les horodatages Wikimedia sont en UTC ; mettre "Europe/Paris" pour l'heure française
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")   # moins de bruit dans docker logs

Spark 4.2.0 -> org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0


## 2. Source Kafka et mise en forme des événements

In [3]:
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")   # listener INTERNAL du réseau Docker
    .option("subscribe", "wikistreams")
    .option("startingOffsets", "earliest")               # chaque requête relit tout le topic
    .load()
)

In [4]:
wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),      # epoch Unix en secondes
    StructField("type", StringType(), True),         # edit, new, log, categorize
    StructField("wiki", StringType(), True),         # ex. frwiki
    StructField("server_name", StringType(), True),  # ex. fr.wikipedia.org
    StructField("namespace", IntegerType(), True),   # 0 = article, 1 = discussion, 2 = utilisateur...
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True),
    ]), True),
])

parsed_df = (
    raw_df
    .select(from_json(col("value").cast("string"), wikimedia_schema).alias("data"))
    .select("data.*")
)

# Colonnes dérivées, réutilisées par toutes les explorations
events_df = (
    parsed_df
    # vrai type timestamp (from_unixtime de la démo renvoie une chaîne)
    .withColumn("event_time", timestamp_seconds(col("timestamp")))
    # variation de taille en octets ; length.old est absent pour une création de page
    .withColumn("delta", col("length.new") - coalesce(col("length.old"), lit(0)))
    .withColumn("taille_abs", _abs(col("delta")))
    .withColumn(
        "classe_taille",
        when(col("taille_abs").isNull(), "0 - sans taille")
        .when(col("taille_abs") < 100, "1 - petite (<100 o)")
        .when(col("taille_abs") < 1000, "2 - moyenne (100-999 o)")
        .otherwise("3 - grosse (>=1000 o)"),
    )
)
events_df.printSchema()

root
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- comment: string (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: integer (nullable = true)
 |    |-- new: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- delta: integer (nullable = true)
 |-- taille_abs: integer (nullable = true)
 |-- classe_taille: string (nullable = false)



## 3. Fonctions utilitaires pour lancer / arrêter les requêtes

In [5]:
def arreter(nom):
    for q in spark.streams.active:
        if q.name == nom:
            q.stop()


def arreter_tout():
    for q in spark.streams.active:
        print("arrêt de", q.name)
        q.stop()


def lancer_console(df, nom, mode="complete", trigger="20 seconds", nb_lignes=50):
    # Résultat écrit dans la console du driver -> docker logs -f pyspark_notebook
    arreter(nom)
    return (
        df.writeStream
        .queryName(nom)
        .format("console")
        .outputMode(mode)
        .option("truncate", False)
        .option("numRows", nb_lignes)
        .trigger(processingTime=trigger)
        .start()
    )


def lancer_memoire(df, nom, mode="complete", trigger="20 seconds"):
    # Résultat écrit dans une table en mémoire nommée `nom`, interrogeable avec spark.sql(...)
    arreter(nom)
    return (
        df.writeStream
        .queryName(nom)
        .format("memory")
        .outputMode(mode)
        .trigger(processingTime=trigger)
        .start()
    )

## Démo d'origine — modifications par heure, robots vs humains

Fenêtre **fixe** de 1 h, mode `complete` (toute la table de résultats est réécrite à chaque micro-batch).

In [6]:
windowed_counts_df = (
    events_df
    .groupBy(window(col("event_time"), "1 hour"), col("bot"))
    .agg(count("*").alias("edit_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
    )
    .orderBy("window_start", "bot")
)
arreter_tout()
q_demo = lancer_console(windowed_counts_df, "demo_bots_1h")

## A. Fenêtre fixe (tumbling) de 10 minutes

`window(event_time, "10 minutes")` découpe le temps en tranches **qui ne se chevauchent pas** : 10:00–10:10, 10:10–10:20…
Chaque modification appartient à **une seule** fenêtre.

`countDistinct` n'est pas supporté en streaming : on utilise `approx_count_distinct` (HyperLogLog).

In [7]:
activite_10min = (
    events_df
    .groupBy(window("event_time", "10 minutes").alias("w"), "bot")
    .agg(
        count("*").alias("nb_modifs"),
        approx_count_distinct("user").alias("contributeurs"),
        _sum(when(col("delta") > 0, col("delta")).otherwise(0)).alias("octets_ajoutes"),
        _sum(when(col("delta") < 0, -col("delta")).otherwise(0)).alias("octets_retires"),
        _sum(when(col("minor"), 1).otherwise(0)).alias("mineures"),
    )
    .select(
        col("w.start").alias("debut"), col("w.end").alias("fin"), "bot", "nb_modifs",
        "contributeurs", "octets_ajoutes", "octets_retires", "mineures",
    )
    .orderBy("debut", "bot")
)
arreter_tout()
q_a = lancer_console(activite_10min, "A_fixe_10min")

arrêt de demo_bots_1h


## B. Fenêtre glissante (overlapping) : largeur 10 min, pas de 5 min

`window(event_time, "10 minutes", "5 minutes")` crée une nouvelle fenêtre toutes les 5 minutes : 10:00–10:10, 10:05–10:15, 10:10–10:20…
Les fenêtres **se chevauchent** : chaque modification est comptée dans 10 / 5 = **2 fenêtres**. On obtient une moyenne mobile plus lisse.

In [8]:
glissante_10_5 = (
    events_df
    .groupBy(window("event_time", "10 minutes", "5 minutes").alias("w"))
    .agg(
        count("*").alias("nb_modifs"),
        _sum(when(col("bot"), 1).otherwise(0)).alias("dont_bots"),
        _round(avg("delta"), 1).alias("delta_moyen"),
        _round(avg("taille_abs"), 1).alias("taille_abs_moy"),
        _max("taille_abs").alias("taille_abs_max"),
    )
    .withColumn("pct_bots", _round(100 * col("dont_bots") / col("nb_modifs"), 1))
    .select(
        col("w.start").alias("debut"), col("w.end").alias("fin"), "nb_modifs", "dont_bots",
        "pct_bots", "delta_moyen", "taille_abs_moy", "taille_abs_max",
    )
    .orderBy("debut")
)
arreter_tout()
q_b = lancer_console(glissante_10_5, "B_glissante_10min_5min")

arrêt de A_fixe_10min


### B'. Vérification : fixe vs glissante

Les deux agrégations écrivent dans des tables en mémoire, puis on compare en SQL.
Résultat attendu : environ **2 fois plus** de fenêtres et un total compté **2 fois plus grand** côté glissante (à une fenêtre près aux deux bords).

In [9]:
arreter_tout()
q_cmp1 = lancer_memoire(events_df.groupBy(window("event_time", "10 minutes")).count(), "cmp_fixe")
q_cmp2 = lancer_memoire(events_df.groupBy(window("event_time", "10 minutes", "5 minutes")).count(), "cmp_glissante")
q_cmp1.processAllAvailable()   # attend que les messages déjà présents dans Kafka soient traités
q_cmp2.processAllAvailable()

spark.sql('''
    SELECT 'fixe 10 min' AS fenetre, COUNT(*) AS nb_fenetres, SUM(count) AS total_comptes FROM cmp_fixe
    UNION ALL
    SELECT 'glissante 10 min / pas 5 min', COUNT(*), SUM(count) FROM cmp_glissante
''').show(truncate=False)

arrêt de B_glissante_10min_5min
+----------------------------+-----------+-------------+
|fenetre                     |nb_fenetres|total_comptes|
+----------------------------+-----------+-------------+
|fixe 10 min                 |23         |4603         |
|glissante 10 min / pas 5 min|47         |9206         |
+----------------------------+-----------+-------------+



## C. Taille des modifications

**C1** — répartition par classe de taille (|nouvelle taille − ancienne taille| en octets), fenêtre fixe de 15 min.

In [10]:
classes_15min = (
    events_df
    .groupBy(window("event_time", "15 minutes").alias("w"), "classe_taille")
    .agg(count("*").alias("nb_modifs"), _sum("taille_abs").alias("octets_touches"))
    .select(col("w.start").alias("debut"), "classe_taille", "nb_modifs", "octets_touches")
    .orderBy("debut", "classe_taille")
)
arreter_tout()
q_c1 = lancer_console(classes_15min, "C1_classes_taille_15min")

arrêt de cmp_fixe
arrêt de cmp_glissante


**C2** — requête **sans état** (ni agrégation ni fenêtre) : alerte sur les très grosses modifications.
Mode `append` : chaque micro-batch n'affiche que les nouveaux événements qui passent le filtre.

In [11]:
SEUIL_OCTETS = 5000
grosses_modifs = (
    events_df
    .filter(col("taille_abs") >= SEUIL_OCTETS)
    .select(
        "event_time", "server_name", "title", "user", "bot", "delta",
        substring("comment", 1, 50).alias("commentaire"),
    )
)
arreter_tout()
q_c2 = lancer_console(grosses_modifs, "C2_alertes_grosses_modifs", mode="append")

arrêt de C1_classes_taille_15min


## D. Pages les plus modifiées — top 3 par fenêtre fixe de 30 min

Le streaming calcule les comptes par (fenêtre, page) dans une table en mémoire ;
le classement « top 3 par fenêtre » se fait ensuite en SQL classique (relancer la 2ᵉ cellule pour rafraîchir).

In [12]:
pages_30min = (
    events_df
    .groupBy(window("event_time", "30 minutes").alias("w"), "title")
    .agg(count("*").alias("nb_modifs"), approx_count_distinct("user").alias("contributeurs"))
    .select(col("w.start").alias("debut"), "title", "nb_modifs", "contributeurs")
)
arreter_tout()
q_d = lancer_memoire(pages_30min, "pages_30min")
q_d.processAllAvailable()

arrêt de C2_alertes_grosses_modifs


In [13]:
spark.sql('''
    SELECT debut, rang, title, nb_modifs, contributeurs FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY debut ORDER BY nb_modifs DESC, title) AS rang
        FROM pages_30min
    ) WHERE rang <= 3
    ORDER BY debut, rang
''').show(50, truncate=False)

+-------------------+----+------------------------------------------------------+---------+-------------+
|debut              |rang|title                                                 |nb_modifs|contributeurs|
+-------------------+----+------------------------------------------------------+---------+-------------+
|2026-10-01 05:30:00|1   |Wikipédia:Contributions rémunérées                    |7        |1            |
|2026-10-01 05:30:00|2   |Yves Béhar                                            |4        |1            |
|2026-10-01 05:30:00|3   |Duncan                                                |3        |1            |
|2026-10-01 06:00:00|1   |Affaire conclue                                       |7        |1            |
|2026-10-01 06:00:00|2   |Odette Polo                                           |6        |1            |
|2026-10-01 06:00:00|3   |Solenzara (fleuve)                                    |5        |1            |
|2026-10-01 06:30:00|1   |Sphinx de Spáta     

## E. Fenêtre glissante 1 h / pas 15 min + watermark, mode `update`

- **watermark de 30 min** : Spark garde l'état d'une fenêtre jusqu'à 30 min après sa fin (en temps d'événement), puis l'oublie. Un événement plus en retard que ça est ignoré. Sans watermark, l'état grossit indéfiniment.
- **mode `update`** : chaque micro-batch n'affiche que les fenêtres qui ont changé (pas de tri possible dans ce mode).
- Variante : `mode="append"` affiche chaque fenêtre **une seule fois**, quand elle est définitivement close (watermark dépassé).

In [14]:
glissante_1h = (
    events_df
    .withWatermark("event_time", "30 minutes")
    .groupBy(window("event_time", "1 hour", "15 minutes").alias("w"), "bot")
    .agg(count("*").alias("nb_modifs"), _round(avg("taille_abs"), 1).alias("taille_abs_moy"))
    .select(col("w.start").alias("debut"), col("w.end").alias("fin"), "bot", "nb_modifs", "taille_abs_moy")
)
arreter_tout()
q_e = lancer_console(glissante_1h, "E_glissante_1h_watermark", mode="update")

arrêt de pages_30min


## F. Activité par wiki (fenêtre fixe de 5 min)

Intéressant quand le producteur suit plusieurs langues, par exemple :
`python wikistream_producer_filters.py --wikis fr.wikipedia.org en.wikipedia.org de.wikipedia.org`

In [21]:
par_wiki = (
    events_df
    .groupBy(window("event_time", "5 minutes").alias("w"), "server_name")
    .agg(count("*").alias("nb_modifs"), _sum(when(col("bot"), 1).otherwise(0)).alias("dont_bots"))
    .select(col("w.start").alias("debut"), "server_name", "nb_modifs", "dont_bots")
    .orderBy("debut", col("nb_modifs").desc())
)
arreter_tout()
q_f = lancer_console(par_wiki, "F_par_wiki_5min")

arrêt de F_par_wiki_5min


## Suivi et arrêt

In [19]:
# Requêtes actives et nombre de lignes lues au dernier micro-batch
for q in spark.streams.active:
    p = q.lastProgress or {}
    print(f"{q.name:30} | {q.status['message']:35} | {p.get('numInputRows')} lignes au dernier batch")

F_par_wiki_5min                | Waiting for next trigger            | 4722 lignes au dernier batch


In [17]:
arreter_tout()

arrêt de F_par_wiki_5min


## Récapitulatif

| Notion | Syntaxe | Effet |
|---|---|---|
| Fenêtre fixe (tumbling) | `window(t, "10 minutes")` | tranches contiguës sans chevauchement, 1 événement → 1 fenêtre |
| Fenêtre glissante (sliding) | `window(t, "10 minutes", "5 minutes")` | une fenêtre démarre tous les 5 min, 1 événement → largeur / pas fenêtres |
| Watermark | `withWatermark(t, "30 minutes")` | borne le retard accepté et permet à Spark de purger l'état |
| Mode `complete` | `outputMode("complete")` | toute la table de résultats à chaque batch (tri possible) |
| Mode `update` | `outputMode("update")` | seulement les lignes modifiées |
| Mode `append` | `outputMode("append")` | seulement les nouvelles lignes ; avec agrégation, exige un watermark |